###**We validated the fraud engine using rule-consistency queries, threshold checks, and referential validation between Gold tables to ensure business correctness.**

## **VALIDATE SCORING LOGIC**

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE
    (fraud_score >= 60 AND fraud_flag != 'YES')
 OR (fraud_score BETWEEN 30 AND 59 AND fraud_flag != 'REVIEW')
 OR (fraud_score < 30 AND fraud_flag != 'NO');


-- score >= 60 → YES
-- 30–59 → REVIEW
-- <30 → NO

## **## VALIDATE TRIGGER SOURCE LOGIC**

In [0]:
%sql
select * from dlt_hc_fraud_detection.bronze_silver_gold. gold_fraud_signals


In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE payment_integrity_status='TRIGGERED'
AND paid_amount <= claim_amount * 1.10;

## **VALIDATE REFERENTIAL CONSISTENCY BETWEEN GOLD TABLES**

In [0]:
%sql
SELECT s.provider_id, s.total_claims, COUNT(g.claim_id) suspicious_claims
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_claim_summary s
LEFT JOIN dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals g
  ON s.provider_id = g.provider_id
GROUP BY s.provider_id, s.total_claims
HAVING COUNT(g.claim_id) > s.total_claims;

## **DATA SANITY CHECKS (Business sense checks)**

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE claim_amount < 0 OR paid_amount < 0;

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE paid_amount > claim_amount
AND payment_integrity_status != 'TRIGGERED';

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE claim_amount = 0
AND fraud_flag = 'YES';

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE paid_amount > claim_amount
AND payment_integrity_status != 'TRIGGERED';

In [0]:
%sql
SELECT COUNT(*) AS over_flagged_rows
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE paid_amount <= claim_amount
AND payment_integrity_status = 'TRIGGERED';

In [0]:
%sql
SELECT claim_id,
       provider_name,
       claim_amount,
       paid_amount as Released_amount,
       payment_integrity_status,
       payment_integrity_reason
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE payment_integrity_status = 'TRIGGERED'
ORDER BY paid_amount - claim_amount DESC;

--Here we detect financial leakage where payment exceeded billed value